In [3]:
"""
Script 01_download_wildfire_api_data.py

NASA FIRMS Wildfire API
--------------------------------
This script downloads global wildfire hotspot data from the
NASA FIRMS API for the last 3 days using the
VIIRS NOAA-21 Near Real-Time dataset.

The downloaded data is stored as:
- GeoJSON
- GeoDataFrame

"""

# =========================================================
# 1. IMPORT LIBRARIES
# =========================================================

import requests
import geopandas as gpd
from pathlib import Path
from shapely.geometry import Point


# =========================================================
# 2. CONFIGURATION
# =========================================================

# My personally created NASA FIRMS MAP_KEY (received by e-mail)
# Source:  https://firms.modaps.eosdis.nasa.gov/api/map_key/
MAP_KEY = "cce9957f998431b07b2b2501061fc6aa"

# NASA FIRMS data source
# VIIRS NOAA-21 Near Real-Time
SOURCE = "VIIRS_NOAA21_NRT"

# Area definition
# "world" downloads global wildfire detections
AREA = "world"

# Time range (in days) for wildfire detections
DAY_RANGE = 3

# =========================================================
#CHOOSE EIGHTER 2.1.1 OR 2.1.2 DEPENDING ON WHETHER YOU ARE RUNNING A NOTEBOOK OR A PYTHON SCRIPT

# 2.1.1 DEFINE PROJECT ROOT IN NOTEBOOK

    # Path.cwd() returns the current working directory.
    # Since the notebook is located inside /notebooks,
    # .parent moves one level up to the project root.

PROJECT_ROOT = Path.cwd().parent

# Define output directory:
# GEO876_Project/data/raw
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

# Create directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: print paths for debugging
print("Project root:")
print(PROJECT_ROOT)

print("\nOutput directory:")
print(OUTPUT_DIR)
"""
# 2.1.2 DEFINE PROJECT ROOT (FOR PYTHON SCRIPTS)

    # __file__ returns the location of the current script.

PROJECT_ROOT = Path(__file__).resolve().parents[2]

# Define output directory:
# GEO876_Project/data/raw
OUTPUT_DIR = PROJECT_ROOT / "data" / "raw"

# Create directory if it does not exist
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional: print paths for debugging
print("Project root:")
print(PROJECT_ROOT)

print("\nOutput directory:")
print(OUTPUT_DIR)
#=========================================================
"""
# Output file
OUTPUT_FILE = OUTPUT_DIR / "wildfires_global_3days.geojson"


# =========================================================
# 3. BUILD API URL
# =========================================================

# NASA FIRMS API endpoint
BASE_URL = "https://firms.modaps.eosdis.nasa.gov/api/area/csv"

# Construct request URL
api_url = (
    f"{BASE_URL}/"
    f"{MAP_KEY}/"
    f"{SOURCE}/"
    f"{AREA}/"
    f"{DAY_RANGE}"
)

print("Step 3: Requesting wildfire data from:")
print(api_url)


# =========================================================
# 4. DOWNLOAD DATA
# =========================================================

try:
    response = requests.get(api_url)

    # Raise an exception if request failed
    response.raise_for_status()

    print("Step 4: Data successfully downloaded.")

except requests.exceptions.RequestException as e:
    print("Error while requesting data:")
    print(e)
    raise


# =========================================================
# 5. LOAD CSV RESPONSE INTO GEODATAFRAME
# =========================================================

# Temporary CSV file path
temp_csv = OUTPUT_DIR / "temp_wildfires.csv"

# Save API response temporarily
with open(temp_csv, "w", encoding="utf-8") as file:
    file.write(response.text)

# Load CSV into GeoDataFrame
gdf = gpd.read_file(temp_csv)

print("\nStep 5: Dataset successfully loaded.")
print(f"Number of wildfire detections: {len(gdf)}")


# =========================================================
# 6. CREATE GEOMETRY COLUMN
# =========================================================

# Create point geometries from longitude and latitude
gdf["geometry"] = gdf.apply(
    lambda row: Point(row["longitude"], row["latitude"]),
    axis=1
)

# Convert to GeoDataFrame
gdf = gpd.GeoDataFrame(
    gdf,
    geometry="geometry",
    crs="EPSG:4326" # WGS 84 - World Geodetic System 1984
)

print("\nStep 6: Geometry column successfully created.")


# =========================================================
# 7. INSPECT DATA
# =========================================================

print("\nStep 7: First rows of dataset:")
print(gdf.head())

print("\nAvailable columns:")
print(gdf.columns.tolist())


# =========================================================
# 8. SAVE AS GEOJSON
# =========================================================

gdf.to_file(
    OUTPUT_FILE,
    driver="GeoJSON"
)

print(f"\nStep 8: GeoJSON successfully saved to:\n{OUTPUT_FILE}")


# =========================================================
# 9. CLEANUP
# =========================================================

# Remove temporary CSV file
temp_csv.unlink(missing_ok=True)

print("\nStep 9: Temporary files removed.")

Project root:
c:\Users\Damian\OneDrive - Universität Zürich UZH\26 FS\GEO876 - Spatial Programming\GEO876_Project

Output directory:
c:\Users\Damian\OneDrive - Universität Zürich UZH\26 FS\GEO876 - Spatial Programming\GEO876_Project\data\raw
Step 3: Requesting wildfire data from:
https://firms.modaps.eosdis.nasa.gov/api/area/csv/cce9957f998431b07b2b2501061fc6aa/VIIRS_NOAA21_NRT/world/3
Step 4: Data successfully downloaded.

Step 5: Dataset successfully loaded.
Number of wildfire detections: 64486

Step 6: Geometry column successfully created.

Step 7: First rows of dataset:
   latitude longitude bright_ti4  scan track    acq_date acq_time satellite  \
0  51.13712  37.93223     335.84   0.5  0.41  2026-05-15        1       N21   
1  51.15498  37.94199      300.6   0.5  0.41  2026-05-15        1       N21   
2  51.21832  34.62391     315.83  0.41  0.37  2026-05-15        1       N21   
3  51.25893  37.76332     301.13  0.49   0.4  2026-05-15        1       N21   
4   51.2604  37.76379   